# Cross-cell C2 · 三模型冻结对比

从 scBIG 报告目录执行本 notebook。同级目录需有 TxPert 与 PerturbDiff 的正式回执和逐条件指标。三种模型在同一组 128 个 K562 留出扰动、每条件 300 个有序对照与 3,352 个基因上评分。图中每个点代表一个条件。

In [ ]:
import csv
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
runs = [
    (root.parent / "crosscell-c2-txpert", "TxPert"),
    (root.parent / "crosscell-c2-perturbdiff", "PerturbDiff"),
    (root, "scBIG"),
]
pearson = (
    ("txpert_macro_pearson_delta_all", "TxPert Δ Pearson", "↑"),
    ("trishift_pearson_delta_all", "TriShift Δ Pearson", "↑"),
    ("systema_pearson_all", "Systema Pearson", "↑"),
)
metrics = [*pearson, ("mean_mse", "Mean MSE", "↓"), ("retrieval", "Retrieval", "↑")]
summaries, tables = [], []
for folder, _ in runs:
    receipt = json.loads((folder / "result.json").read_text())
    data = (folder / "conditions.jsonl").read_bytes()
    assert hashlib.sha256(data).hexdigest() == receipt["conditions_file_sha256"]
    assert receipt["status"] == "evaluated" and receipt["formal_eligible"] is True
    assert receipt["test_updates"] == 0 and receipt["test_controls_per_condition"] == 300
    table = {}
    for line in data.splitlines():
        row = json.loads(line)
        assert row["condition_id"] not in table
        assert all(value["gene_count"] == 3352 for value in row["results"])
        table[row["condition_id"]] = {
            **{item["metric_id"]: item["value"] for item in row["results"]},
            "mean_mse": row["mean_mse"], "retrieval": row["retrieval"],
        }
    assert len(table) == receipt["conditions"] == 128
    summaries.append(receipt)
    tables.append(table)
identities = ("condition_ids_sha256", "paired_rows_sha256", "test_panel_genes",
              "shared_fit_support_file_sha256")
assert all(len({s[key] for s in summaries}) == 1 for key in identities)
assert all(list(table) == list(tables[0]) for table in tables)
assert summaries[2]["test_panel_genes"] == 3352

columns = ["condition_id"] + [f"{name}__{metric}" for _, name in runs for metric, _, _ in metrics]
with (root / "condition-metrics.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(columns)
    for condition in tables[0]:
        writer.writerow([condition] + [
            table[condition][metric] for table in tables for metric, _, _ in metrics
        ])

colors = ("#566680", "#b47448", "#17666e")
fig, axes = plt.subplots(2, len(metrics), figsize=(14.8, 6.2), layout="constrained")
rng = np.random.default_rng(20)
names = [name for _, name in runs]
for col, (key, title, direction) in enumerate(metrics):
    top, bottom = axes[:, col]
    for idx, (receipt, table, color) in enumerate(zip(summaries, tables, colors, strict=True)):
        values = np.asarray([row[key] for row in table.values()], dtype=float)
        finite = values[np.isfinite(values)]
        if key in {metric for metric, _, _ in pearson}:
            aggregate = next(item for item in receipt["pearson"] if item["metric_id"] == key)
            assert len(finite) == aggregate["finite_condition_count"]
            expected = aggregate["macro_mean"]
        else:
            expected = receipt["retrieval_mean" if key == "retrieval" else key]
        assert np.isclose(finite.mean(), expected)
        top.bar(idx, expected, color=color, width=.65)
        top.annotate(f"{expected:.4f}", (idx, expected), (0, 3 if expected >= 0 else -3),
                     textcoords="offset points", ha="center", fontsize=8,
                     va="bottom" if expected >= 0 else "top")
        bottom.scatter(idx + rng.uniform(-.20, .20, len(finite)), finite, color=color, s=7, alpha=.45)
    top.set_title(f"{title} {direction}", fontsize=10)
    for axis in (top, bottom):
        axis.set_xticks(range(len(runs)), names, rotation=14)
        axis.set_xlim(-.50, len(runs) - .50)
        axis.margins(y=.22)
        axis.grid(axis="y", alpha=.18)
        axis.axhline(0, color="#333333", lw=.65)
axes[0, 0].set_ylabel("Condition macro average")
axes[1, 0].set_ylabel("Each frozen test condition")
fig.savefig(root / "frozen-metrics.png", dpi=175, bbox_inches="tight")
plt.show()